# Análisis SQL — Producción agrícola vs. Deforestación
 
## Objetivo
 
Identificar qué provincias argentinas combinan **alta producción agrícola** (soja, maíz, trigo) 
con **alto nivel de deforestación de bosque nativo**, simulando el tipo de análisis de riesgo que 
hoy exige la regulación europea EUDR a los exportadores.
 
## Preguntas que vamos a responder
 
1. ¿Qué provincias producen más soja/maíz/trigo en total?
2. ¿Qué provincias perdieron más bosque nativo en total?
3. ¿Cómo se relacionan producción y deforestación al cruzar ambas por provincia?
4. ¿Qué provincias clasificarían como "alto riesgo" según ese cruce?
5. ¿Este patrón cambia según el cultivo (soja vs. maíz vs. trigo)?
6. ¿Cómo evolucionó la producción a lo largo de los años en las provincias de mayor riesgo?
 
## Limitación a tener presente
 
El dataset de deforestación es un dato **acumulado por provincia** (sin año), mientras que 
producción sí tiene año. Por eso el cruce principal se hace sobre totales históricos, no 
año a año — es una aproximación geográfica, no una trazabilidad exacta por lote o período.

## Objetivo del proyecto
identificar qué provincias combinan alta producción exportable con alto nivel de deforestación, simulando el tipo de análisis de riesgo que exige la EUDR.

In [11]:
import pandas as pd
import sqlite3

conexion = sqlite3.connect("../database/trazabilidad.db")

## Queries

### 1. Ranking de producción por provincia
¿Qué provincias producen más soja/maíz/trigo en total?

In [12]:
query_1 = """
SELECT provincia, SUM(produccion_tm) as total_produccion FROM produccion
GROUP BY provincia
ORDER BY total_produccion DESC
"""

df_ranking_produccion = pd.read_sql(query_1, conexion) 
pd.read_sql(query_1, conexion)

,provincia,total_produccion
0,Buenos Aires,1141283297
1,Córdoba,804752359
2,Santa Fe,600255082
3,Entre Ríos,162184884
4,Santiago del Estero,128408876
5,La Pampa,98403310
6,Salta,59091252
7,Chaco,53880863
8,San Luis,44047293
9,Tucumán,32333842


Buenos Aires, Córdoba y Santa Fe concentran la mayor parte de la producción histórica de 
soja/maíz/trigo (más de 900M de toneladas cada una) — coincide con la "zona núcleo" agrícola 
conocida. Santiago del Estero, Salta, Chaco y Jujuy, aunque están más abajo en el ranking, son 
las provincias del norte donde se concentra la mayor presión de deforestación, y son las que 
más nos interesan para el análisis de riesgo.

### 2.Ranking de deforestación por provincia
¿Qué provincias perdieron más bosque nativo en total?

In [13]:
query_2 = """
SELECT provincia, SUM(superficie_en_hectáreas) as total_deforestacion FROM deforestacion
GROUP BY provincia
ORDER BY total_deforestacion DESC
"""

df_ranking_deforestacion = pd.read_sql(query_2, conexion)
pd.read_sql(query_2, conexion)

,provincia,total_deforestacion
0,Santiago del Estero,59942.0
1,Formosa,33345.0
2,Chaco,24427.0
3,Salta,13956.0
4,San Luis,9933.0
5,Río Negro,9705.0
6,Chubut,9266.0
7,Entre Ríos,9024.0
8,La Rioja,6989.0
9,Córdoba,5190.0


Santiago del Estero, Formosa, Chaco y Salta encabezan el ranking de deforestación (todas 
provincias del norte argentino). Es exactamente el patrón opuesto al de producción: las 
provincias que más producen (Buenos Aires, Córdoba, Santa Fe) son las que **menos** 
deforestaron, mientras que las que más deforestaron producen relativamente poco en 
comparación. Esto sugiere que el riesgo real no está en las provincias "grandes" del 
ranking de producción, sino en el cruce específico entre ambas variables — que es 
justamente lo que vamos a calcular en la próxima query.

## 3. El cruce principal — producción vs. deforestación
combinar el ranking de producción y el de deforestación en una sola tabla, con una fila por provincia mostrando ambos totales lado a lado

In [14]:
query_3 = """
WITH produccion_total AS (
    SELECT provincia, SUM(produccion_tm) as total_produccion FROM produccion
    GROUP BY provincia
    ORDER BY total_produccion DESC),
deforestacion_total AS (
    SELECT provincia, SUM(superficie_en_hectáreas) as total_deforestacion FROM deforestacion
    GROUP BY provincia
    ORDER BY total_deforestacion DESC)
SELECT p.provincia, p.total_produccion, d.total_deforestacion
FROM produccion_total p
JOIN deforestacion_total d ON p.provincia = d.provincia
"""

df_ranking_produccion_deforestacion = pd.read_sql(query_3, conexion)
pd.read_sql(query_3, conexion)

,provincia,total_produccion,total_deforestacion
0,Buenos Aires,1141283297,1032.0
1,Córdoba,804752359,5190.0
2,Santa Fe,600255082,3764.0
3,Entre Ríos,162184884,9024.0
4,Santiago del Estero,128408876,59942.0
5,La Pampa,98403310,4423.0
6,Salta,59091252,13956.0
7,Chaco,53880863,24427.0
8,San Luis,44047293,9933.0
9,Tucumán,32333842,1215.0


El cruce con INNER JOIN devuelve 22 provincias (no 23): Tierra del Fuego queda excluida 
porque no tiene registros de soja/maíz/trigo en el dataset — es coherente, esa provincia 
patagónica no produce estos cultivos. No se pierde información relevante para el análisis.

## 4. Clasificación de riesgo
clasificar cada provincia en un nivel de riesgo ("Alto", "Medio", "Bajo")

In [15]:
query_4 = """
WITH produccion_total AS (
    SELECT provincia, SUM(produccion_tm) as total_produccion FROM produccion
    GROUP BY provincia
    ORDER BY total_produccion DESC),
deforestacion_total AS (
    SELECT provincia, SUM(superficie_en_hectáreas) as total_deforestacion FROM deforestacion
    GROUP BY provincia
    ORDER BY total_deforestacion DESC)
SELECT p.provincia, p.total_produccion, d.total_deforestacion,
    CASE
        WHEN d.total_deforestacion >= 20000 THEN 'Alto'
        WHEN d.total_deforestacion > 5000 THEN 'Medio'
        ELSE 'Bajo'
    END AS nivel_riesgo
FROM produccion_total p
JOIN deforestacion_total d ON p.provincia = d.provincia
"""

df_riesgo = pd.read_sql(query_4, conexion)
pd.read_sql(query_4, conexion)

,provincia,total_produccion,total_deforestacion,nivel_riesgo
0,Buenos Aires,1141283297,1032.0,Bajo
1,Córdoba,804752359,5190.0,Medio
2,Santa Fe,600255082,3764.0,Bajo
3,Entre Ríos,162184884,9024.0,Medio
4,Santiago del Estero,128408876,59942.0,Alto
5,La Pampa,98403310,4423.0,Bajo
6,Salta,59091252,13956.0,Medio
7,Chaco,53880863,24427.0,Alto
8,San Luis,44047293,9933.0,Medio
9,Tucumán,32333842,1215.0,Bajo


Con el corte Alto (≥20.000 ha) / Medio (>5.000 ha) / Bajo (≤5.000 ha), 4 provincias quedan 
en riesgo "Alto": Santiago del Estero, Chaco, Formosa (todas del norte) — y sorprendentemente 
también aparece en Medio riesgo Córdoba, Entre Ríos, Salta, San Luis, Río Negro, Chubut y 
La Rioja. El hallazgo clave: Buenos Aires y Santa Fe (las provincias con MAYOR producción) 
son "Bajo" riesgo, mientras que Formosa (producción relativamente baja) es "Alto" riesgo — 
confirma que el riesgo de deforestación NO está correlacionado con el volumen de producción, 
sino con la región geográfica específica.

## 5. Desagregado por cultivo
ver si el patrón de riesgo cambia según el cultivo.

In [16]:
query_5 = """
WITH produccion_total AS (
    SELECT provincia,cultivo, SUM(produccion_tm) as total_produccion FROM produccion
    GROUP BY provincia,cultivo
    ORDER BY total_produccion DESC),
deforestacion_total AS (
    SELECT provincia, SUM(superficie_en_hectáreas) as total_deforestacion FROM deforestacion
    GROUP BY provincia
    ORDER BY total_deforestacion DESC)
SELECT p.provincia,p.cultivo, p.total_produccion, d.total_deforestacion,
    CASE
        WHEN d.total_deforestacion >= 20000 THEN 'Alto'
        WHEN d.total_deforestacion > 5000 THEN 'Medio'
        ELSE 'Bajo'
    END AS nivel_riesgo
FROM produccion_total p
JOIN deforestacion_total d ON p.provincia = d.provincia
ORDER BY p.cultivo,d.total_deforestacion DESC
"""

df_riesgo_cultivo = pd.read_sql(query_5, conexion)
pd.read_sql(query_5, conexion)

,provincia,cultivo,total_produccion,total_deforestacion,nivel_riesgo
0,Santiago del Estero,maíz,59813813,59942.0,Alto
1,Formosa,maíz,2811605,33345.0,Alto
2,Chaco,maíz,19544260,24427.0,Alto
3,Salta,maíz,27030987,13956.0,Medio
4,San Luis,maíz,32982923,9933.0,Medio
5,Río Negro,maíz,88146,9705.0,Medio
6,Chubut,maíz,330,9266.0,Medio
7,Entre Ríos,maíz,52791352,9024.0,Medio
8,La Rioja,maíz,666,6989.0,Medio
9,Córdoba,maíz,347920217,5190.0,Medio


Al abrir el cruce por cultivo, el patrón de riesgo se mantiene consistente: Santiago del 
Estero, Formosa y Chaco quedan en "Alto" riesgo para los TRES cultivos (soja, maíz, trigo) 
por igual — no es un problema exclusivo de la soja, como muchas veces se asume en el debate 
público. Dato interesante: Chaco produce más maíz que soja (19,5M vs. 30,2M tn — en realidad 
al revés, más soja), y en las provincias de alto riesgo el maíz también tiene peso relevante, 
no solo la soja.

## 6.Evolución de producción en provincias de alto riesgo
traer la producción por año, filtrando solo esas 3 provincias, para después graficarlo.

In [18]:
query_6 = """
    SELECT provincia,anio, SUM(produccion_tm) as total_produccion FROM produccion
    WHERE provincia in ('Santiago del Estero','Chaco','Formosa')
    GROUP BY provincia, anio
    ORDER BY anio
"""

df_evolucion = pd.read_sql(query_6, conexion) 
pd.read_sql(query_6, conexion)

,provincia,anio,total_produccion
0,Chaco,1969,63060
1,Formosa,1969,15750
2,Santiago del Estero,1969,103326
3,Chaco,1970,98875
4,Formosa,1970,30800
...,...,...,...
163,Formosa,2023,64048
164,Santiago del Estero,2023,5056609
165,Chaco,2024,737992
166,Formosa,2024,93403


Serie completa 1969-2024 para Santiago del Estero, Chaco y Formosa. Santiago del Estero 
muestra un salto muy grande: de ~103.000 tn en 1969 a ~5.000.000 tn en 2023-2024 — un 
crecimiento de casi 50x, que coincide con la expansión de la frontera agrícola hacia el 
norte del país en las últimas décadas. Este dato refuerza la relación entre expansión 
agrícola y presión sobre el bosque nativo en esa zona.